# PINN notebook (runnable)

This notebook demonstrates a minimal Physics-Informed Neural Network pipeline with:
- Data I/O helpers (CSV or synthetic stress data) and prediction export.
- Stress transforms assembling x = [|S_pre|, S_pre].
- MLP backbone with data and PDE heads.
- Adaptive loss balancing via learned log-variances.
- End-to-end training and inference cells ready to run in Jupyter.


In [ ]:
import math
from pathlib import Path
from typing import Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## Data I/O
- Load from CSV with columns: sin_theta, cos_theta, Sxx, Sxy, Syy.
- Or create synthetic data.
- Each sample returns theta, S_pre (Sxx, Sxy, Syy), and a target L_hat.


In [ ]:
class StressDataset(Dataset):
    def __init__(self, csv_path: Optional[str] = None, n_samples: int = 512, noise: float = 0.01):
        super().__init__()
        if csv_path and Path(csv_path).exists():
            df = pd.read_csv(csv_path)
            required_cols = ['sin_theta', 'cos_theta', 'Sxx', 'Sxy', 'Syy']
            missing = set(required_cols) - set(df.columns)
            if missing:
                raise ValueError(f'Missing columns: {missing}')
            data = df[required_cols].to_numpy(dtype=np.float32)
            sin_theta, cos_theta, Sxx, Sxy, Syy = data.T
            theta = np.arctan2(sin_theta, cos_theta)
        else:
            theta = torch.linspace(-math.pi, math.pi, n_samples)
            Sxx = torch.sin(theta) * 0.6 + noise * torch.randn_like(theta)
            Sxy = torch.cos(theta) * 0.3 + noise * torch.randn_like(theta)
            Syy = 0.2 * torch.sin(2 * theta) + noise * torch.randn_like(theta)
            theta = theta.numpy()
            Sxx, Sxy, Syy = [t.numpy() for t in (Sxx, Sxy, Syy)]

        L_hat = 0.5 * Sxx + 0.3 * Sxy + 0.2 * Syy

        self.theta = torch.tensor(theta, dtype=torch.float32)
        self.S_pre = torch.stack([
            torch.tensor(Sxx, dtype=torch.float32),
            torch.tensor(Sxy, dtype=torch.float32),
            torch.tensor(Syy, dtype=torch.float32),
        ], dim=1)
        self.L_hat = torch.tensor(L_hat, dtype=torch.float32)

    def __len__(self):
        return len(self.theta)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {
            'theta': self.theta[idx],
            'S_pre': self.S_pre[idx],
            'L_hat': self.L_hat[idx],
        }


def save_predictions(path: str, inputs: torch.Tensor, outputs: Dict[str, torch.Tensor]) -> None:
    df = pd.DataFrame(inputs.cpu().numpy(), columns=['|S_pre|', 'Sxx_pre', 'Sxy_pre', 'Syy_pre'])
    for key, value in outputs.items():
        df[key] = value.detach().cpu().numpy()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'Saved predictions to {path}')


## Stress transforms
- Build rotation matrices.
- Pack [Sxx, Sxy, Syy] into 2x2 matrices.
- Assemble x = [|S_pre|, S_pre] used by the network.


In [ ]:
def rotation_matrix(theta: torch.Tensor) -> torch.Tensor:
    c, s = torch.cos(theta), torch.sin(theta)
    T = torch.stack([
        torch.stack([c, -s], dim=-1),
        torch.stack([s, c], dim=-1),
    ], dim=-2)
    return T


def rotate_stress(S: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    T = rotation_matrix(theta)
    return T @ S @ T.transpose(-2, -1)


def pack_stress(S_components: torch.Tensor) -> torch.Tensor:
    Sxx, Sxy, Syy = S_components.unbind(-1)
    return torch.stack([
        torch.stack([Sxx, Sxy], dim=-1),
        torch.stack([Sxy, Syy], dim=-1),
    ], dim=-2)


def stress_norm(S_components: torch.Tensor) -> torch.Tensor:
    Sxx, Sxy, Syy = S_components.unbind(-1)
    return torch.sqrt(Sxx**2 + 2 * Sxy**2 + Syy**2 + 1e-8)


def assemble_input(theta: torch.Tensor, S_pre: torch.Tensor):
    S_matrix = pack_stress(S_pre)
    S_rot = rotate_stress(S_matrix, theta)
    x = torch.cat([stress_norm(S_pre).unsqueeze(-1), S_pre], dim=-1)
    return x, S_rot


## Model
Shared MLP backbone with two heads:
- L_pred: supervised term.
- residual: PDE term driven toward zero.


In [ ]:
class StressPINN(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dim: int = 128, dropout: float = 0.05):
        super().__init__()
        layers = []
        dims = [input_dim, hidden_dim, hidden_dim, hidden_dim]
        for din, dout in zip(dims[:-1], dims[1:]):
            layers.extend([nn.Linear(din, dout), nn.GELU(), nn.Dropout(dropout)])
        self.backbone = nn.Sequential(*layers)
        self.head_L = nn.Linear(hidden_dim, 1)
        self.head_pde = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.backbone(x)
        L_pred = self.head_L(h).squeeze(-1)
        residual = self.head_pde(h).squeeze(-1)
        return {'L_pred': L_pred, 'residual': residual}


## Loss and training loop
Adaptive log-variance weights balance data and PDE terms.


In [ ]:
class AdaptiveMultiTaskLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))

    def forward(self, losses: Tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        balanced = []
        for i, loss in enumerate(losses):
            balanced.append(torch.exp(-self.log_vars[i]) * loss + self.log_vars[i])
        return sum(balanced)


def train_epoch(model: StressPINN, criterion: AdaptiveMultiTaskLoss, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total = 0.0
    for batch in loader:
        theta = batch['theta'].to(device)
        S_pre = batch['S_pre'].to(device)
        target = batch['L_hat'].to(device)

        x, _ = assemble_input(theta, S_pre)
        preds = model(x.to(device))

        data_loss = torch.mean((preds['L_pred'] - target) ** 2)
        pde_loss = torch.mean(preds['residual'] ** 2)
        loss = criterion((data_loss, pde_loss))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(theta)
    return total / len(loader.dataset)


In [ ]:
# Training example
train_ds = StressDataset(csv_path=None, n_samples=512, noise=0.01)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

model = StressPINN().to(device)
criterion = AdaptiveMultiTaskLoss().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(criterion.parameters()), lr=3e-3, weight_decay=1e-4)

num_epochs = 5
for epoch in range(1, num_epochs + 1):
    loss = train_epoch(model, criterion, train_loader, optimizer)
    print(f"Epoch {epoch:02d} - loss: {loss:.4f} - log_vars: {criterion.log_vars.data.tolist()}")


## Inference and export
Generate predictions and save them to CSV.


In [ ]:
model.eval()
with torch.no_grad():
    sample = next(iter(DataLoader(train_ds, batch_size=16, shuffle=False)))
    theta = sample['theta'].to(device)
    S_pre = sample['S_pre'].to(device)
    x, S_rot = assemble_input(theta, S_pre)
    preds = model(x)

save_predictions('artifacts/pinn_predictions.csv', x, {'L_pred': preds['L_pred'], 'residual': preds['residual']})
Path('artifacts/pinn_predictions.csv').read_text().splitlines()[:5]
